### PyTorch Lightning basic classification model ###

In [38]:
import sys
import os
import copy
import pandas as pd
import numpy as np
from pathlib import Path
import albumentations as alb
from matplotlib import pyplot as plt

# PyTorch
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision.models import resnet50, ResNet50_Weights

# Lightning module
from lightning.pytorch import LightningModule, Trainer

# Appearance of the Notebook
from IPython.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

# Import this module with autoreload
%load_ext autoreload
%autoreload 2

import computervision as cv
from computervision.fileutils import FileOP
from computervision.imageproc import ImageData, is_image
from computervision.transformations import AugmentationTransform
from computervision.datasets import DatasetFromDF

# Print version info
print(f'Package version: {cv.__version__}')
print(f'Authors:         {cv.__authors__}')
print(f'Python version:  {sys.version}')

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Package version: v0.0.2
Authors:         The Core for Computational Biomedicine at Harvard Medical School
https://dbmi.hms.harvard.edu/about-dbmi/core-computational-biomedicine
Python version:  3.12.3 (main, Jun 18 2025, 17:59:45) [GCC 13.3.0]


In [39]:
data_dir = os.environ.get('DATA_DIR')
print(f'data_dir: {data_dir}')

# Directory where the data are
dataset_name = 'dataset_dental_roboflow'
dataset_dir = os.path.join(data_dir, 'roboflow')
image_dir = os.path.join(dataset_dir, dataset_name, 'cropped')

# Set up a model directory for the trained model
model_dir = os.path.join(data_dir, 'model')
Path(model_dir).mkdir(exist_ok=True, parents=True)

data_dir: /app/data


### Load the annotations ###

In [8]:
annotations_file_name = 'annotations_cropped_dset.parquet'
annotations_file = os.path.join(image_dir, annotations_file_name)
df = pd.read_parquet(annotations_file)

file_col = 'file_name'
bbox_col= 'bbox'
label_col = 'label'
dset_col = 'dset'

labels = sorted(list(df[label_col].unique()))
label2id = dict(zip(labels, range(len(labels))))
id2label = {category_id: label for label, category_id in label2id.items()}
display(id2label)

# Now we can add a category id to the data frame
df = df.assign(category=df[label_col].apply(lambda label: label2id.get(label)))
display(df.head())

# Check the images
file_list = [os.path.join(image_dir, file_name) for file_name in df[file_col].unique()]
checked = [is_image(file) for file in file_list]
assert len(file_list) == sum(checked), f'WARNING: Could not open all {len(file_list)} images at: {image_dir}'
print(f'Image directory:        {image_dir}')
print(f'Total number of images: {len(file_list)}')
print(f'Annotations:            {df.shape[0]}')

{0: 'Calculus',
 1: 'amalgam',
 2: 'caries',
 3: 'composite',
 4: 'root filling',
 5: 'tooth'}

,multi_file,file_name,width,height,area,label,bbox,pos,pos_bbox,dset,category
0,a0ab0bec5c.jpg,tooth_02_a0ab0bec5c.jpg,152,236,35872,tooth,None,[2],None,train,5
1,5c6869fdf0.jpg,amalgam_03_5c6869fdf0.jpg,427,265,113155,amalgam,"[182, 29, 119, 112]","[19, 20]","[[143, 20, 284, 245], [20, 33, 382, 232]]",train,1
2,96a9546c1e.jpg,caries_08_96a9546c1e.jpg,525,253,132825,caries,"[97, 97, 46, 32]","[19, 20]","[[129, 21, 396, 232], [20, 20, 256, 233]]",train,2
3,1e36413c9e.jpg,composite_05_1e36413c9e.jpg,535,241,128935,composite,"[67, 32, 169, 87]","[29, 30]","[[213, 44, 322, 197], [20, 20, 341, 221]]",train,3
4,872cd0d6f2.jpg,tooth_15_872cd0d6f2.jpg,213,181,38553,tooth,None,[15],None,train,5


Image directory:        /app/data/roboflow/dataset_dental_roboflow/cropped
Total number of images: 7550
Annotations:            7550


### Image augmentations for training and validation/testing ###

In [20]:
# Initial scaling and padding for the bigger dimension
max_image_size = 640

# Model input size
im_width, im_height = 224, 224
train_transforms = AugmentationTransform(im_width=im_width, im_height=im_height).\
                get_transforms(name='train_transform')

# Resize and then normalize 
# with ImageNet mean and standard deviation for ResNet50
image_net_mean = [0.485, 0.456, 0.406]
image_net_std = [0.229, 0.224, 0.225]

# This transform is essential and needs to be applies for both training and validation
resize_and_normalize = [alb.Resize(width=im_width, height=im_height),
                        alb.Normalize(mean=image_net_mean, std=image_net_std)]
train_transforms.extend(resize_and_normalize)
train_transform = alb.Compose(train_transforms)

# However, for validation and testing, we don't want the augmentations, 
# so we just resize and normalize the data
test_transform = alb.Compose(resize_and_normalize)

### Datasets from the annotations data frame ###

In [24]:
# Create the data sets from the data frame
train_dataset = DatasetFromDF(data=df.loc[df[dset_col] == 'train'],
                              image_dir=image_dir,
                              file_name_col=file_col,
                              label_id_col='category',
                              max_image_size=max_image_size,
                              transform=train_transform,
                              validate=True)

### Pick a model ###

The ResNet50Model class implements a variation of the ResNet50 architecture, which is a well-known type of convolutional neural networks particularly suitable for image classification tasks.

A list of available models is here: https://pytorch.org/vision/stable/models.html#classification

The ResNet50 model: https://pytorch.org/vision/stable/models/generated/torchvision.models.resnet50.html#torchvision.models.resnet50

In [28]:
class ResNet50Model:
    """ This is the ResNet50 model from torchvision.models """
    def __init__(self, n_outputs):
        self.n_outputs = n_outputs
    def create_model(self):
        model = resnet50(weights=ResNet50_Weights.DEFAULT)
        model.fc = nn.Sequential(
            nn.Linear(in_features=model.fc.in_features, out_features=512),
            nn.ReLU(),
            nn.Linear(in_features=512, out_features=self.n_outputs)
        )
        return model

### PyTorch Lightning Framework ###

In [41]:
class DentalModel(LightningModule):
    def __init__(self,
                 train_dataset,
                 batch_size,
                 num_workers=1,
                 lr=1.0e-3,
                 model=None):
        super().__init__()
        self.save_hyperparameters(ignore=['model'])
        self.train_dataset = train_dataset
        self.batch_size = batch_size
        self.num_workers = num_workers
        self.lr = lr
        self.decimals = 5
        # Model architecture
        if model is None:
            self.model = ResNet50Model().create_model()
        else:
            self.model = model
        # Loss function
        self.criterion = nn.CrossEntropyLoss()

    def train_dataloader(self):
        dl = DataLoader(self.train_dataset,
                        batch_size=self.batch_size,
                        num_workers=self.num_workers,
                        shuffle=True,
                        pin_memory=True)
        return dl

    def forward(self, x, *args, **kwargs):
        x = self.model(x)
        return x

    def training_step(self, batch, batch_idx, *args, **kwargs):
        image, label = batch
        pred = self.forward(image)
        loss = self.criterion(pred, label)
        return loss

    def predict_step(self, batch, batch_idx, *args, **kwargs):
        image, label = batch
        output = self.forward(image)
        return output

    def configure_optimizers(self):
        opt = torch.optim.AdamW(self.parameters(), lr=self.lr)
        return opt

### Instantiate a model ###

In [43]:
categories = sorted(list(df['category'].unique()))
print(categories)

resnet_model = ResNet50Model(n_outputs=len(categories)).create_model()

model = DentalModel(train_dataset=train_dataset,
                   batch_size=16,
                   num_workers=2,
                   model=resnet_model)

[0, 1, 2, 3, 4, 5]


### Train the model ###

In [44]:
# Create the trainer object and train the model for 5 epochs
# Train for at least 40 epochs to get good results

max_epochs = 5
dental_model_trainer = Trainer(max_epochs=max_epochs,
                              deterministic=True,
                              default_root_dir=model_dir)
# Run the training
dental_model_trainer.fit(model)

💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name      | Type             | Params | Mode 
-------------------------------------------------------
0 | model     | ResNet           | 24.6 M | train
1 | criterion | CrossEntropyLoss | 0      | train
-------------------------------------------------------
24.6 M    Trainable params
0         Non-trainable params
24.6 M    Total params
98.241    Total estimated model params size (MB)
155       Modules in train mode
0         Modules in eval mode


Training: |          | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_epochs=5` reached.


In [45]:
# Save the model in a Hugging Face compatible format
checkpoint_file = os.path.join(model_dir, 'model.ckpt')
dental_model_trainer.save_checkpoint(checkpoint_file)